In [1]:
# prescriptive analytics (Fraud Decision Engine)


In [2]:
from Feature_eng import add_interactions # Bring in feature function python file 
import joblib # Model 
import pandas as pd # load data 

In [3]:
model = joblib.load('best_model') # Model
df = pd.read_csv('fraud_risk_dataset.csv') # raw data

In [4]:
# Seperate features and target 
X = df.drop(columns = ['fraud', 'transaction_id']) # features 
y = df['fraud'] # target variable 

df['fraud_risk_score'] = model.predict_proba(X)[:,1] # Generate predicited probability and turn it into a risk score column 
df['fraud_risk_score'].head()

0    0.458961
1    0.442526
2    0.456741
3    0.466550
4    0.452103
Name: fraud_risk_score, dtype: float64

In [8]:
# Define thresholds (These are policy cutoffs)
low_risk = 0.28 # This is a point where fraud becomes less apparent
high_risk = 0.65 # This is a point where fraud becomes more apparent



In [9]:
def assign_risk_tier(p): # Define risk tier func (p represent probability between 0-1)
    if p >= high_risk: # If model is very confident this is fraud, label high
        return "high"
    
    elif p >= low_risk: # This is the gray zone, tranasctions are suspcious but uncertain. Transaction should be placed on hold and futher review is required. 
        return 'medium'
    
    else:               # Low predicted proability, safe enough to approve
        return 'low'
    


    



In [10]:
df['risk_tier'] = df['fraud_risk_score'].apply(assign_risk_tier) # Apply function to the dataset (for each value risk tier is outputted and stored in a new column)

In [11]:
# risk tier distribution 

df['risk_tier'].value_counts()

risk_tier
medium    4942
high        58
Name: count, dtype: int64

In [ ]:
# Model doesn't seem confident in prediciting risk tiers. No low risk tier is prevelant which is a red flag in fraud due to the fact it is impossible, to rveiew every single transactions.
# Moving foward, let's inspect the score distribution... This will show us where score actually live nd where natural cut off points exist 


In [12]:
df['fraud_risk_score'].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]) # distribution 

count    5000.000000
mean        0.467327
std         0.045442
min         0.377875
1%          0.402482
5%          0.419829
10%         0.430581
25%         0.444374
50%         0.454248
75%         0.468082
90%         0.524016
95%         0.537265
99%         0.655278
max         0.769542
Name: fraud_risk_score, dtype: float64

In [13]:
# Instead of fixed number we can define tier by volumn control 
# Example low -> bottom 80% ---- medium -> next 15% ----- high -> top 5%

low_cut = df['fraud_risk_score'].quantile(0.80) # bottom 10% of score
high_cut = df['fraud_risk_score'].quantile(0.95) # top 10% of score 

def assign_risk_tier(p):
    if p >= high_cut:
        return 'high'
    elif p >= low_cut:
        return 'medium'
    else: 
        return 'low'

df['risk_tier'] = df['fraud_risk_score'].apply(assign_risk_tier)
df['risk_tier'].value_counts(normalize = True)


# This guarantees....
# Review load is finite
# auto approval exists 
# high risk is meaningful 

risk_tier
low       0.80
medium    0.15
high      0.05
Name: proportion, dtype: float64

In [ ]:
# Decision Layer 
def fraud_action(tier):
    if tier == "HIGH":
        return "BLOCK"
    elif tier == "MEDIUM":
        return "REVIEW"
    else:
        return "ALLOW"
    

In [15]:
# Apply decicsion layer(actions) to the dataset as a new column 

df['decision'] = df['fraud_risk_score'].apply(fraud_action)
df['decision'].value_counts()

decision
ALLOW    5000
Name: count, dtype: int64

In [17]:
df

,transaction_id,customer_age,account_tenure_months,region,transaction_amount,merchant_category,payment_method,device_type,num_transactions_24h,international,previous_chargebacks,fraud,fraud_risk_score,risk_tier,decision
0,100000,65,68,South,130.13,Travel,Digital Wallet,Tablet,4,0,0,0,0.458961,low,ALLOW
1,100001,22,4,South,28.27,Travel,Debit,Tablet,1,0,0,0,0.442526,low,ALLOW
2,100002,43,53,South,304.83,Travel,Credit,Tablet,2,0,0,0,0.456741,low,ALLOW
3,100003,72,69,West,39.87,Restaurants,Debit,Tablet,1,0,1,1,0.466550,low,ALLOW
4,100004,21,8,South,125.73,Gas,Debit,Desktop,3,0,0,0,0.452103,low,ALLOW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,104995,49,69,Northeast,88.01,Online Services,Digital Wallet,Tablet,0,0,1,0,0.430338,low,ALLOW
4996,104996,27,136,Midwest,25.96,Restaurants,Credit,Mobile,1,0,1,0,0.443199,low,ALLOW
4997,104997,19,77,Northeast,197.17,Restaurants,Debit,Mobile,2,0,1,0,0.448196,low,ALLOW
4998,104998,49,62,Northeast,88.32,Gas,Debit,Desktop,1,0,0,0,0.448148,low,ALLOW


In [ ]:
# There's an issue here... The distribution in probabilites is very small
# This means most of the scores fit very close together causing each probability to be deemed as low
# In otherwords, all scores are the same. This is caused by the model random forest 
# Random forest doesn't do well on probability calibration but is great for ranking 
# This is not a faliure, it's a expected outcome 
# The fix would be to use ranking based percentiles forcing seperation even when ties are close 

In [ ]:
# Let's begin a ranking based risk stratgey

# Assign each row a ranked position 
df['risk_percentile'] = df['fraud_risk_score'].rank(
    method = 'average', pct = True) # averages out duplicates and turns rankings into percentilese

df['risk_tier'] = pd.cut(
    df['risk_percentile'], # bins numeric columns into intervals
    bins = [0, 0.80, 0.95, 1.0], 
    labels = ['low', 'medium', 'high'],
    include_lowest= True    #ensure lowest percentile is include
)

df['risk_tier'].value_counts(normalize= True)

risk_tier
low       0.80
medium    0.15
high      0.05
Name: proportion, dtype: float64

In [53]:
# decision layer 
def assign_risk_tier(tier):
    if tier == 'high':
        return 'BLOCK'
    elif tier == 'low':
        return 'ALLOW'
    else:
          return 'Review'
    
# inspect result
df['decision'].value_counts()


decision
ALLOW     4000
REVIEW     750
BLOCK      250
Name: count, dtype: int64

In [73]:
# Prescriptive Evaluation 

# Fraud rate by risk tier
tier_eval = pd.crosstab(
    df['risk_tier'],
    df['fraud'],
    normalize = 'index'
)

tier_eval.columns = ['Legit %', 'Fraud%']
tier_eval

,Legit %,Fraud%
risk_tier,,
low,0.964500,0.035500
medium,0.818667,0.181333
high,0.660000,0.340000


In [67]:
#Fraud rate by decision 
decision_eval = pd.crosstab(
    df['decision'], 
    df['fraud'],
    normalize = 'index')


decision_eval.columns = ['Legit %', 'Fraud %']
decision_eval

,Legit %,Fraud %
decision,,
ALLOW,0.964500,0.035500
REVIEW,0.818667,0.181333
BLOCK,0.660000,0.340000


In [72]:
# fraud capture rate 
fraud_capture_rate = (df[df['fraud'] == 1]['decision'].isin(
    ['REVIEW', 'BLOCK'])
    .mean()
)

fraud_capture_rate

np.float64(0.6088154269972452)

In [ ]:
# 61% of fraudualent transactions are being intercpeted. 